In [2]:
for chapter_name, chapter_content in leaves.items():
    print(chapter_name)
    print("========")
    print(len(chapter_content))

Luxembourgish / Summary
1474
Luxembourgish / Keywords
1474
1. Introduction
1758
2. Sketch of the Sociohistorical and Sociolinguistic
1527
Evolution / [COMP: FIGURE
1527
Evolution / 1
1527
Evolution / HERE]
10915
3. Phonetics and Phonology / 3.1 Syllable and Word Structure, Prosody
12811
4. Morphosyntax / 4.1 Inflection of the Noun
5050
4. Morphosyntax / 4.2 Adjective Inflection
3619
4. Morphosyntax / 4.3 Articles / )
1646
4. Morphosyntax / 4.3 Articles / een
1646
4. Morphosyntax / 4.3 Articles / engem
1646
4. Morphosyntax / 4.3 Articles / keen
1646
4. Morphosyntax / 4.4 Personal Pronouns
3160
4. Morphosyntax / 4.5 Possession and Partitives / däers
1819
4. Morphosyntax / 4.5 Possession and Partitives / där
1819
4. Morphosyntax / 4.5 Possession and Partitives / där/der
1819
4. Morphosyntax / 4.5 Possession and Partitives / däers/es
1819
4. Morphosyntax / 4.6 Prepositions
1661
4. Morphosyntax / 4.7 Verbs
16295
5. Selected Syntactic Characteristics
1483
6. Lexical Structures
10500
7. Langu

In [3]:
sample_text = leaves["4. Morphosyntax / 4.7 Verbs"]

In [ ]:
import os
import time
import math
import re
import json
from typing import List, Dict, Any, Tuple

import tiktoken  # pip install tiktoken
from openai import OpenAI  # pip install openai

# -------- Config --------
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")  
TEMPERATURE = float(os.environ.get("OPENAI_TEMPERATURE", "0.0"))

MAX_TOKENS_PER_CHUNK = int(os.environ.get("MAX_TOKENS_PER_CHUNK", "300"))
CHUNK_OVERLAP_TOKENS = int(os.environ.get("CHUNK_OVERLAP_TOKENS", "50"))

SYSTEM_PROMPT = (
    "You are a precise extractor that outputs only valid JSON according to the given schema."
)

USER_PROMPT_TEMPLATE = """You will receive a passage from a Luxembourgish grammar book. Please extract all core grammar points from the text.

For each grammar point, provide:
- Rule: a concise and detailed description of the grammar rule.
- Action: the corrective or usage instruction related to the rule.
- Example: include all example sentences from the text (leave empty if none).

Return the result strictly in the following JSON format (no extra explanations, no additional text):

{
  "grammar_points": [
    {
      "rule": "string",
      "action": "string",
      "example": "string or list of strings"
    }
  ]
}

Here is the text:
\"\"\"{text}\"\"\"
"""



# -------- Tokenizer helpers (for chunking) --------
def get_tokenizer(model_name: str = "gpt-4o-mini"):
    # 兼容 o/gpt 系列编码，退回 cl100k_base
    try:
        return tiktoken.encoding_for_model(model_name)
    except Exception:
        return tiktoken.get_encoding("cl100k_base")

enc = get_tokenizer(MODEL)

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def split_paragraphs(text: str) -> List[str]:
    # 基于空行/多换行断段
    paras = re.split(r"\n{2,}", text.strip())
    paras = [p.strip() for p in paras if p.strip()]
    return paras

def split_sentences(paragraph: str) -> List[str]:
    # 简易句子分割：按句末标点 + 换行/空格
    # 更严格可替换为 nltk/pysbd
    parts = re.split(r"(?<=[.!?。！？])\s+", paragraph.strip())
    return [s.strip() for s in parts if s.strip()]

def semantic_chunks(text: str,
                    max_tokens: int = MAX_TOKENS_PER_CHUNK,
                    overlap_tokens: int = CHUNK_OVERLAP_TOKENS) -> List[str]:

    paragraphs = split_paragraphs(text)
    chunks: List[str] = []
    current: List[str] = []
    current_tokens = 0

    def flush():
        nonlocal current, current_tokens
        if current:
            chunks.append(" ".join(current).strip())
            current = []
            current_tokens = 0

    for para in paragraphs:
        sents = split_sentences(para)
        for s in sents:
            stoks = count_tokens(s)
            # 如果单句超限，硬切句（极少见）
            if stoks > max_tokens:
                # 退化为按字符近似分片
                char_step = max(200, math.floor(len(s) * max_tokens / max(stoks, 1)))
                for i in range(0, len(s), char_step):
                    piece = s[i:i+char_step]
                    if piece.strip():
                        current.append(piece)
                        current_tokens += count_tokens(piece)
                        if current_tokens >= max_tokens:
                            flush()
                continue

            if current_tokens + stoks <= max_tokens:
                current.append(s)
                current_tokens += stoks
            else:
                flush()
                current.append(s)
                current_tokens = stoks
        # 段落结束不强制 flush，让下一段句子填满

    flush()

    # 应用重叠
    if overlap_tokens > 0 and len(chunks) > 1:
        overlapped: List[str] = []
        prev_tail_tokens: List[int] = []  # 记录上块尾部句子的 tokens
        for idx, ch in enumerate(chunks):
            if idx == 0:
                overlapped.append(ch)
                # 计算尾部句子 tokens 便于后续拼接
                prev_tail_tokens = [count_tokens(s) for s in split_sentences(ch)]
            else:

                tail_sents = split_sentences(chunks[idx-1])
                acc = []
                acc_tok = 0
                for s in reversed(tail_sents):
                    t = count_tokens(s)
                    if acc_tok + t > overlap_tokens:
                        break
                    acc.append(s)
                    acc_tok += t
                acc = list(reversed(acc))
                if acc:
                    merged = " ".join(acc + [ch])
                    overlapped.append(merged)
                else:
                    overlapped.append(ch)
        chunks = overlapped

    # 去重与清洗
    dedup = []
    seen = set()
    for ch in chunks:
        k = ch.strip()
        if k and k not in seen:
            dedup.append(k)
            seen.add(k)
    return dedup

# -------- OpenAI call with retries --------
client = OpenAI(api_key=OPENAI_API_KEY)

def call_openai_extract(chunk: str,
                        model: str = MODEL,
                        temperature: float = TEMPERATURE,
                        max_retries: int = 5,
                        initial_backoff: float = 1.0) -> Dict[str, Any]:
    prompt = USER_PROMPT_TEMPLATE.format(chunk=chunk)
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=temperature,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                response_format={"type": "json_object"}, 
            )
            content = resp.choices.message.content
            return json.loads(content)
        except Exception as e:
            sleep_s = initial_backoff * (2 ** attempt) + 0.1 * (attempt)
            time.sleep(sleep_s)
            if attempt == max_retries - 1:
                raise

def normalize_result(obj: Dict[str, Any]) -> Dict[str, Any]:
    gps = obj.get("grammar_points", [])
    if isinstance(gps, str):
        gps = [gps]
    if gps is None:
        gps = []
    gps = [str(x).strip() for x in gps if str(x).strip()]

    ex = obj.get("example", "")
    if ex is None:
        ex = ""
    ex = str(ex).strip()

    return {"grammar_points": gps, "example": ex}

def merge_results(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    # 合并去重 grammar_points；优先保留第一个非空 example
    merged_points = []
    seen = set()
    example = ""
    for r in results:
        for p in r.get("grammar_points", []):
            if p not in seen:
                merged_points.append(p)
                seen.add(p)
        if not example and r.get("example", ""):
            example = r["example"]
    return {"grammar_points": merged_points, "example": example}

def extract_grammar_from_text(text: str) -> Dict[str, Any]:
    chunks = semantic_chunks(text, MAX_TOKENS_PER_CHUNK, CHUNK_OVERLAP_TOKENS)
    results = []
    for i, ch in enumerate(chunks, 1):
        obj = call_openai_extract(ch)
        results.append(normalize_result(obj))
    final_obj = merge_results(results)
    return final_obj


result = extract_grammar_from_text(sample_text)
print(json.dumps(result, ensure_ascii=False, indent=2))
